# **Memory (Checkpointing) in LangGraph**

By default, LangGraph graphs are stateless — each `invoke()` starts fresh.
**Checkpointing** persists state between invocations using a `thread_id`, enabling multi-turn conversations.

Key piece: `InMemorySaver` — an in-process checkpoint store (swap for `PostgresSaver` in production).

In [ ]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os

load_dotenv()

if os.environ.get('GROQ_API_KEY'):
    print("Groq API Key is set.")
else:
    raise ValueError("GROQ_API_KEY not found.")

In [ ]:
llm = ChatGroq(model="llama3-8b-8192", temperature=0)

### **Step-1: Define The Schema**
`add_messages` is a built-in LangGraph reducer that deduplicates and appends messages by `id`, so each invocation only needs to return new messages.

In [ ]:
from typing import TypedDict, List, Annotated
from langgraph.graph.message import add_messages

class graph_schema(TypedDict):
    # Annotated[List, add_messages] means LangGraph calls add_messages(old_list, new_list)
    # to merge state across invocations — it handles deduplication automatically
    messages: Annotated[List, add_messages]

### **Step-2: Node Function**

In [ ]:
def welcome(schema: graph_schema) -> graph_schema:

    curr_messages = schema['messages']

    # Pass the full message history to the LLM for context-aware replies
    response = llm.invoke(curr_messages).content

    schema['messages'] = f"Your message was {curr_messages}. Here's my response: {response}"

    return schema

### **Step-3: Build the Graph**

In [ ]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(graph_schema)
graph.add_node("welcome", welcome)
graph.add_edge(START, "welcome")
graph.add_edge("welcome", END)

### **Step-4: Compile With a Checkpointer**
`checkpointer=checkpoint` enables persistence. Every `invoke()` that shares the same `thread_id` picks up where the last one left off.

In [ ]:
from IPython.display import Image
from langgraph.checkpoint.memory import InMemorySaver

# InMemorySaver stores checkpoints in RAM — fine for notebooks, lost on restart
checkpoint = InMemorySaver()

memory_graph = graph.compile(checkpointer=checkpoint)
Image(memory_graph.get_graph().draw_mermaid_png())

### **Step-5: Run With a Config**
`configurable.thread_id` is the conversation key. Two calls with the same `thread_id` share memory.

In [ ]:
from langchain_core.messages import HumanMessage

# First turn: introduce yourself
config = {'configurable': {'thread_id': 'user_123'}}

response1 = memory_graph.invoke(
    {"messages": [HumanMessage(content="Hi, my name is Ansh.")]},
    config
)
for message in response1['messages']:
    message.pretty_print()

In [ ]:
# Second turn — same thread_id, so the graph remembers the first turn
response2 = memory_graph.invoke(
    {"messages": [HumanMessage(content="What is my name?")]},
    config   # same config = same thread = graph has memory of previous exchange
)
for message in response2['messages']:
    message.pretty_print()